# Code Lab - An Entire RAG Pipeline

En esta sección vamos a crear un **RAG pipeline** completo y desde cero, que servirá como base para profundizar en los capítulos posteriores. En este capítulo se incluirá un pipeline con las siguientes características:

- Vincular un LLM con una cuenta de OpenAI
- Instalación de paquetes de Python
- Web crawling, división de documentos y embedding chunks para la indexación de datos
- Búsqueda por vectores similares (vector similarity search)
- Generar respuestas integrando contexto a los prompts
- Sin interfaz 

En primer lugar instalamos e importamos las librerías necesarias:

In [2]:
# %pip install langchain_community langchain_experimental langchain-ollama langchainhub chromadb langchain beautifulsoup4

In [3]:
import os
from langchain_community.document_loaders import WebBaseLoader
import bs4
import ollama
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain import hub
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_experimental.text_splitter import SemanticChunker

USER_AGENT environment variable not set, consider setting it to identify your requests.


---

### **1. Indexing**

Ahora viene la primera fase del RAG, **indexing**, donde obtendremos los datos del prompt del usuario, les haremos un pre-procesamiento y los vectorizaremos. En este pequeño apartado haremos lo siguiente:

- Web loading y web crawling
- Dividir (splitting) los datos en chunks para que el algoritmo de vectorización de *Chroma* sea más eficiente
- Convertir los chunks en embeddings
- Añadir los chunks y embeddings a la *vector store* de *Chroma*

#### 1.1 Web Loading y Web Crawling

El contenido lo vamos a sacar de la siguiente página:

In [4]:
webPage = 'https://lilianweng.github.io/posts/2023-06-23-agent/'

loader = WebBaseLoader(
    web_path = webPage,
    bs_kwargs = dict(
        parse_only = bs4.SoupStrainer(
            class_ = ('post-title', 'post-header', 'post-content')
        )
    ),
)

docs = loader.load()


De esta manera podemos obtener el contenido de una web como documentos.

*WebBaseLoader* hace mucho trabajo:

1. Petición HTTP a la URL que hemos especificado
2. Hace un parseo del HTML son *BeautifulSoup*, parseando únicamente los elementos incluidos en *parse_only*
3. Extrae el texto del parseo
4. Crea objetos tipo *document* con el contenido de la web que hemos extraído (que se guardan en la variable docs en este caso)

Una vez hecho esto pasamos al siguiente paso, splitting

#### 1.2 Splitting

En este paso simplemente vamos a dividir el documento que hemos obtenido anteriormente en distintos *chunks*, para reducir tiempo de procesamiento, convertiéndolos en textos mucho más manejables sin perder la coherencia de cada *chunk*. 

En nuestro caso vamos a usar *SemanticChunker* aunque hay otras muchas opciones:

In [5]:
embeddings = OllamaEmbeddings(model='nomic-embed-text')
text_splitter = SemanticChunker(embeddings)
splits = text_splitter.split_documents(docs)
print(len(splits))

21


Vemos que *SemanticChunker* nos divide el documento en 21 chunks. Esto se hace en función del contexto de cada chunk, en lugar de proporcionar una longitud establecida. Esto en general nos hace una mejor división del texto al ser semántica (por ello necesita un modelo como *OllamaEmbeddings*), pero es más pesado computacionalmente.

Vamos a probar a ver un Chunker diferente, el cual si se basa en tamaño concreto, conocido como *RecursiveCharacterTextSplitter*, uno de los splitters más usados de langchain.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter_size = RecursiveCharacterTextSplitter(
    chunk_size = 500,   # Tamaño máximo de cada chunk
    chunk_overlap = 50,   # Indica cuantos caracteres se repiten entre un chunk y el siguiente (para no perder contexto)
    separators = ['\n\n', '\n', ' ', '']   # Lista ordenada de los separadores (intenta separar por parrafos, si no cabe en chunk_size prueba por lineas, etc.)
)

splits_size = text_splitter.split_documents(docs)
print(len(splits_size))

21


Y vemos que el tiempo en este caso es mucho menor, ya que estamos indicando el tamaño de los chunks y la separación es mucho más simple, pero la calidad de los chunks baja considerablemente. 

Hemos que decidir entre calidad o eficiencia, aunque en este caso, vamos a optar por calidad ya que no hay mucho texto.

#### 1.3 Embeddings y Vector Store de Chroma

A continuación vamos a crear la *vector store* con *Chroma* y vamos a guardar los *embeddings* de nuestro texto. Esto lo podemos hacer muy sencillamente con *Chroma* en *Python*:

In [7]:
vector_store = Chroma.from_documents(
    documents = splits,
    embedding = OllamaEmbeddings(model='nomic-embed-text')
)

retriever = vector_store.as_retriever()

Internamente, el método *Chroma.from_documents()* está haciendo lo siguiente:

1. Itera sobre cada *Document* en la variable *splits*
2. Para cada *Document*, usa el embedding, en este caso *OllamaEmbeddings()* para generar el vector
3. Guarda el texto original y su correspondiente vector en la *vector store* de *Chroma*

Podemos ver que los embeddings se estan generando correctamente con el siguiente código (no forma dentro del código final):

In [8]:
data = vector_store.get(include=['embeddings'])

# Para ver los embeddings del primer chunk, no los imprimimos todos (hay 768 xd)
print(data['embeddings'][0][:30])

[ 0.02725698  0.05026644 -0.13049266 -0.07393885  0.03244552 -0.00505756
  0.02465732 -0.00350863 -0.02056746 -0.0068397  -0.01521997 -0.0046853
  0.10491869  0.03440397 -0.01472879  0.01154074  0.01415378 -0.07126766
 -0.00486376  0.01106874  0.03104531 -0.01692409  0.02466148 -0.01964626
  0.01312184  0.00217164  0.01849631 -0.04251862 -0.01740811 -0.00742486]


El *retriever* que se vio anteriormente, se utilizará para el algoritmo de *vector similarity* en nuestra *vector store*, ya que nos proporciona los métodos necesarios para ello.

Podemos ver un ejemplo de su uso, que nuevamente no forma parte del código oficial:

In [9]:
query = 'How does RAG compare with fine-tuning?'
relevant_docs = retriever.get_relevant_documents(query)
relevant_docs

/tmp/ipykernel_7643/3719435863.py:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  relevant_docs = retriever.get_relevant_documents(query)


[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Observation: ...'),
 Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='4. ... 5. ... Constraints:\n1. ~4000 word limit for short term memory. Your short term memory is short, so immediately save important information to files. 2. If you are unsure how you previously did something or want to recall past events, thinking about similar events will help you remember. 3.'),
 Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='2023). In both experiments on knowledge-intensive tasks and decision-making tasks, ReAct works better than the Act-only baseline where Thought: … step is removed. Reflexion (Shinn & Labash 2023) is a framework to equip agents with dynamic memory and self-reflection capabilities to improve reasoning skills. Reflexion has a standard RL setup, in which the reward model provide

Y este resultado es la información más relevante de nuestra *vector store* que se asemeja más al prompt. Sin embargo, esto es simplemente un ejemplo muy sencillo, ya que no hemos dado una respuesta, solo la información que es similar. Para ello pasamos a la siguiente sección...

---

### **2. Retriever y Generation**

Los pasos que vamos a seguir en la fase de *retriever* y *generation* son:

- Obtener la user query
- Vectorizar dicha user query
- Hacer un *similarity search* con la *vector store* para encontrar tanto los vectores más relacionados con el input del usuario como su contenido. Esto es de lo que se encarga el **retriever**
- Pasar el contenido obtenido por el *retriever* a un *template*. A esto se conoce como **hydrating**
- Pasar el *hydrated prompt* al LLM
- Presentar la respuesta del LLM al usuario

In [19]:
rag_prompt = hub.pull('jclemens24/rag-prompt')
print(rag_prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'jclemens24', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '1a1f3ccb9a5a92363310e3b130843dfb2540239366ebe712ddd94982acc06734'} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


Este template lo hemos obtenido directamente de *Langchain Hub*, obteniendo así el prompt que le pasaremos al LLM.

Vemos que este template requiere de un *context* y una *question*, lo cual proporcionaremos más adelante, completando el proceso de **hydrating** correctamente. Este *prompt template* es una parte fundamental del RAG, ya que nos permite comunicarnos con el LLM correctamente. 

No es solo un string, sino un contexto junto con el prompt del usuario para que el LLM proporcione la mejor respuesta posible.

Ahora vamos a definir una función para obtener todo el contenido de los documentos que almacenamos en la *vector store*:

In [28]:
def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

Perfecto, ahora el último paso antes de montar la chain (*Langchain chain*) es definir el LLM que vamos a usar:

In [25]:
llm = ChatOllama(model='qwen3:8b', temperature=0)

Ahora vamos a montar la *chain*. Esta *chain* la vamos a definir en un formato en específico llamado LCEL, ya que hace que el código sea más legible y compacto, y abre nuevas técnicas para optimizar la velocidad y eficiencia del código

In [ ]:
rag_chain = (
    {'context': retriever | format_docs,
     'question': RunnablePassthrough()}
        | rag_prompt
        | llm
        | StrOutputParser()
)

Esta *chain* representa una cadena de operaciones usando el framework de *Langchain*.

Al poner en uno de los values del dict un `Runnable`, *rag_chain* es un objeto de la clase `RunnableParallel` que devuelve 

    { 'context': <texto>, 'question': <texto> }

donde ambos `context` y `question` reciben el mismo input de la *chain*, pero cada uno lo procesa de forma distinta:

- En **`context`** el input pasa por `retriever | format_docs`, por lo que buscamos los vectores similares en la vector store y los devolvemos como documentos, para pasarlos por la función y obtener el string

- En **`question`** se devuelve el input del usuario sin modificar gracias a `RunnablePassthrough()`

Cabe destacar que `|` no es el típico `OR` de *Python*, sino que conecta el *Runnable* de la izquierda (*retriever*) como input a la función de la derecha (*format_docs*)

---

Al usar el operador `|` estamos construyendo un objeto de la clase **`RunnableSequence`**, donde cada etapa toma la salida de la anterior y la pasa a la siguiente. En concreto:

1. La primera etapa es el **`RunnableParallel`** (el dict), que construye el diccionario `{context, question}`

2. Esa salida entra al **`prompt`** (el cual es un template de Langchain), que inserta `context` y `question` en el template

3. El resultado se envía al **`llm`**, que genera la respuesta

4. Finalmente, **`StrOutputParser()`** pasa la salida del LLM a un string normal y corriente

---

Por lo que solo queda llamar al método `invoke` de *rag_chain* para pasarle el input del usuario y generar la respuesta, pasando por todo el proceso que acabamos de explicar

In [32]:
from IPython.display import Markdown

response = rag_chain.invoke('What are the advantajes of using RAG?')

display(Markdown(response))

<think>
Okay, the user is asking about the advantages of using RAG (Retrieval-Augmented Generation). Let me start by recalling what RAG is. RAG combines retrieval of information from a database with a generative model to answer questions. The context provided mentions long-term memory management, GPT-3.5 agents, and constraints like a 4000-word limit for short-term memory. 

First, I need to list the advantages based on the context. The context talks about long-term memory management, which suggests that RAG can handle persistent data storage. Then there's delegation of tasks using GPT-3.5 agents, which might imply efficiency in task distribution. The constraints mention saving important info to files, so maybe RAG helps in organizing data efficiently.

Wait, the user's question is about advantages, so I should focus on benefits. The context doesn't explicitly list advantages, but I can infer them. For example, RAG allows for up-to-date information by retrieving data, which is better than static models. It also enables handling large volumes of data through memory management. The use of agents for delegation could mean better task efficiency. Also, the ability to save important info to files might relate to data persistence and accessibility.

But the user provided a context that seems to be part of a code structure, not directly about RAG advantages. Maybe the context is from a system that uses RAG, and the user wants to know the advantages based on that system's features. The context mentions constraints like short-term memory limits, so RAG's ability to offload to long-term memory is an advantage. Also, the system's architecture with agents and memory management could be part of RAG's benefits.

I need to make sure I'm not missing any key points. The answer should list advantages like improved accuracy with up-to-date data, efficient memory management, scalability, and task delegation. Also, the ability to handle large data volumes by using retrieval mechanisms. The context's constraints about saving to files might relate to data persistence, which is another advantage.

Wait, the user's context seems to be part of a code structure, possibly for a system that uses RAG. The answer should tie the advantages to the features mentioned in the context. For example, long-term memory management is an advantage, allowing the system to retain information beyond short-term limits. Delegation of tasks using agents could mean better resource management. The 4000-word limit for short-term memory implies that RAG helps in managing data efficiently by offloading to long-term storage.

I should structure the answer by listing each advantage, explaining how it relates to the context provided. Make sure to connect each point to the features mentioned, like memory management, task delegation, and data persistence. Also, note that the system's architecture allows for these advantages through its design elements.
</think>

The advantages of using Retrieval-Augmented Generation (RAG) are closely tied to its architecture and the constraints outlined in the context. Here's a detailed breakdown of the benefits, aligned with the provided system design:

---

### **Advantages of Using RAG**
1. **Enhanced Accuracy with Up-to-Date Information**  
   RAG combines retrieval of external data (e.g., databases, documents) with generative models, ensuring answers are grounded in the latest and most relevant information. This is critical for systems requiring dynamic data, such as real-time analytics or knowledge bases.

2. **Efficient Long-Term Memory Management**  
   The system's long-term memory (e.g., file storage) allows persistent data retention beyond short-term memory limits (~4000 words). This ensures critical information is saved for future use, avoiding data loss and enabling scalable knowledge accumulation.

3. **Task Delegation via GPT-3.5 Agents**  
   By delegating simple tasks to GPT-3.5-powered agents, the system optimizes resource allocation. This reduces the computational load on the main model, improving efficiency and enabling parallel processing of tasks.

4. **Scalability and Flexibility**  
   The architecture supports modular components (e.g., retrieval, generation, memory management), allowing the system to scale horizontally. For example, adding more retrieval sources or agents can enhance capacity without overhauling the core framework.

5. **Cost-Effective Resource Utilization**  
   The system's constraints (e.g., short-term memory limits) enforce efficiency. By offloading data to long-term storage and delegating tasks, the system minimizes redundant computations and reduces operational costs.

6. **Improved Contextual Understanding**  
   Retrieval mechanisms allow the model to access external context, enabling it to handle complex queries that exceed its training data. This is particularly useful for domains requiring specialized knowledge (e.g., legal, medical).

7. **Self-Reflection and Optimization**  
   The system's design encourages continuous evaluation of actions (e.g., performance metrics, memory usage). This self-critique loop ensures the system adapts to new challenges and refines its strategies over time.

---

### **Core Architecture Implementation**
Below is the code structure for the system, implementing the described advantages:

#### **1. `memory_manager.py`**  
*Manages short-term and long-term memory, ensuring data persistence.*

```python
FILENAME
```python
class MemoryManager:
    def __init__(self, short_term_limit=4000):
        self.short_term_memory = []
        self.long_term_memory = []
        self.short_term_limit = short_term_limit

    def save_to_long_term(self, data):
        """Saves critical information to long-term storage."""
        self.long_term_memory.append(data)
        # Simulate file saving
        with open("long_term_memory.txt", "a") as f:
            f.write(str(data) + "\n")

    def retrieve_from_long_term(self, query):
        """Retrieves data from long-term memory."""
        return [item for item in self.long_term_memory if query in item]

    def manage_short_term(self, data):
        """Manages short-term memory, offloading to long-term if needed."""
        if len(self.short_term_memory) >= self.short_term_limit:
            # Save oldest data to long-term memory
            self.save_to_long_term(self.short_term_memory.pop(0))
        self.short_term_memory.append(data)
```

#### **2. `agent_delegation.py`**  
*Delegates simple tasks to GPT-3.5 agents for efficiency.*

```python
FILENAME
```python
class TaskAgent:
    def __init__(self, model="gpt-3.5"):
        self.model = model

    def execute_task(self, task):
        """Delegates a task to the agent."""
        # Simulate task execution
        print(f"Executing task with {self.model}: {task}")
        return f"Result of {task}"

    def delegate(self, task):
        """High-level task delegation."""
        return self.execute_task(task)
```

#### **3. `retrieval_augmentation.py`**  
*Integrates retrieval and generation for dynamic responses.*

```python
FILENAME
```python
class RetrievalAugmentor:
    def __init__(self, memory_manager, agent):
        self.memory_manager = memory_manager
        self.agent = agent

    def augment(self, query):
        """Retrieves relevant data and generates a response."""
        retrieved_data = self.memory_manager.retrieve_from_long_term(query)
        if retrieved_data:
            return f"Retrieved data: {retrieved_data}. Final answer: {self.agent.delegate(query)}"
        return self.agent.delegate(query)
```

#### **4. `main.py`**  
*Entrypoint for the system, integrating all components.*

```python
FILENAME
```python
def main():
    # Initialize components
    memory_manager = MemoryManager()
    agent = TaskAgent()
    augmentor = RetrievalAugmentor(memory_manager, agent)

    # Example usage
    query = "What is the capital of France?"
    response = augmentor.augment(query)
    print(response)

if __name__ == "__main__":
    main()
```

---

### **Key Design Decisions**
- **Modularity**: Each component (memory, agents, retrieval) is decoupled, enabling independent scaling and updates.
- **Persistence**: Long-term memory is saved to files, ensuring data retention across sessions.
- **Efficiency**: Short-term memory limits force the system to prioritize critical data, avoiding overload.
- **Adaptability**: The system can be extended with additional agents or retrieval sources without rewriting core logic.

This architecture ensures RAG's advantages are fully realized, balancing accuracy, efficiency, and scalability.